In [87]:
import os
from datetime import UTC, datetime, timedelta

import cabaret
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits

In [117]:
## Observatory components

site = cabaret.Site(
    sky_background=1000,  # e-/m^2/arcsec^2/s
    seeing=1.35,  # arcseconds
    elevation=2500,  # meters
)

telescope = cabaret.Telescope(
    focal_length=8,  # meters
    diameter=1,  # meters
)

pixel_defects = {
    "hot": {"type": "constant", "value": 14000 * 5.09, "rate": 0.5 / 100, "seed": 42},
    "hot2": {"type": "constant", "value": 14100 * 5.09, "rate": 0.5 / 100, "seed": 44},
    "cold": {"type": "constant", "value": 300 * 5.09, "rate": 0.3 / 100, "seed": 84},
    "qe": {
        "type": "quantum_efficiency_map",
        "quantum_efficiency_std": 1 / 100,
        "seed": 168,
    },
}
camera = cabaret.Camera(
    name="Simulated SPIRIT",
    width=1024,  # pixels
    height=1280,  # pixels
    bin_x=1,  # pixels
    bin_y=1,  # pixels
    pitch=12,  # microns
    max_adu=2**14 - 1,  # ADU
    well_depth=50000,  # electrons
    bias=2700,  # ADU
    gain=5.09,  # electrons per ADU
    read_noise=90,  # electrons
    dark_current=150,  # electrons per second
    average_quantum_efficiency=1,  # fraction
    rotation=0,  # degrees
    pixel_defects=pixel_defects,
)

In [118]:
observatory = cabaret.Observatory(
    name="Simulated SPECULOOS",
    site=site,
    telescope=telescope,
    camera=camera,
)

camera.set_plate_scale_from_focal_length(telescope.focal_length)

In [119]:
if_pds = 1 if pixel_defects is not None else 0
print(f"Pixel defects: {'enabled' if if_pds else 'disabled'}")

Pixel defects: enabled


In [120]:
seed = -1
n_imgs = 100
n_imgs_cal = 10
exp_time = 10  # seconds
readout_time = 0.1  # seconds
ra = 30  # degrees
dec = 0  # degrees
image_run = f"{ra}_{dec}_s{seed}_n{n_imgs}_nc{n_imgs_cal}_{exp_time}s_r{readout_time}s_pds{if_pds}"

filter_band = cabaret.Filters.J
dateobs_0 = datetime(2024, 6, 1, 0, 0, 0, tzinfo=UTC)

center = SkyCoord(ra=ra, dec=dec, unit="deg")

sources = cabaret.GaiaQuery.get_sources(
    center=center,
    radius=camera.get_fov_radius() * 1.5,
    filter_band=filter_band,
    limit=10000,
    dateobs=dateobs_0,
    # timeout=60,
)


In [121]:
sources

Sources(coords=<SkyCoord (ICRS): (ra, dec) in deg
    [(30.01256964,  0.0987736 ), (30.01421626,  0.01043707),
     (29.91970803,  0.0137681 ), (30.07406508,  0.02438499),
     (29.93283002,  0.0681009 ), (29.99658582, -0.05248145),
     (30.05099602,  0.01520318), (29.97900401,  0.09661455),
     (30.01664428,  0.00326507), (30.02042012,  0.0292388 ),
     (29.97125829, -0.00844537), (29.967707  ,  0.02424007),
     (30.03422885, -0.04716269), (29.95609298,  0.05462417),
     (29.97153866,  0.01172373), (29.91059541,  0.01132927),
     (30.02071977, -0.08170214), (29.96517825, -0.04679088),
     (29.98281542, -0.06719352), (29.94688729, -0.02774925),
     (30.04787476, -0.04410528), (30.00346146,  0.07008257),
     (30.05446076,  0.06966225), (29.95860436, -0.020835  ),
     (30.0723209 , -0.0323222 ), (30.0103893 , -0.04351454),
     (30.03211944,  0.01519599), (29.90591886, -0.03880758),
     (29.94139285, -0.00028372), (30.02735585, -0.03424667),
     (29.93012167,  0.06180883), (3

In [122]:
# brightest sources
# find index of brightest sources.fluxes
brightest_indices = np.argsort(sources.fluxes)[::-1][:10]
sources[brightest_indices]


Sources(coords=<SkyCoord (ICRS): (ra, dec) in deg
    [(30.01256964,  0.0987736 ), (30.01421626,  0.01043707),
     (29.91970803,  0.0137681 ), (30.07406508,  0.02438499),
     (29.93283002,  0.0681009 ), (29.99658582, -0.05248145),
     (30.05099602,  0.01520318), (29.97900401,  0.09661455),
     (30.01664428,  0.00326507), (30.02042012,  0.0292388 )]>, fluxes=array([354322.78 ,  50280.46 ,  36224.215,  25246.385,  18171.838,
        17955.56 ,  17369.965,  14079.885,  11184.052,   9545.511],
      dtype=float32))

In [123]:
# Light frames
os.makedirs(f"data/{image_run}", exist_ok=True)
# Generate the FITS image for this filter
for j in range(n_imgs):
    dateobs = dateobs_0 + timedelta(seconds=j * (exp_time + readout_time))

    ## Light frame
    hdu = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=exp_time,  # seconds
        dateobs=dateobs,
        sources=sources,
        fwhm_multiplier=7,  # to determine the rendering radius around each star
        user_header={
            "FILTER": filter_band.name,
            "IMAGETYP": "Light Frame",
            "object": image_run,
        },
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu.writeto(
        f"data/{image_run}/{image_run}_{filter_band.name}_Light_{j}.fits",
        overwrite=True,
    )

# Calibration frames (bias, dark, flat)
os.makedirs(f"data/{image_run}", exist_ok=True)
for j in range(n_imgs_cal):
    dateobs = dateobs_0 + timedelta(seconds=j * (exp_time + readout_time))

    ## Bias frame
    hdu_bias = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=0,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "Bias Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_bias[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_bias.writeto(
        f"data/{image_run}/Bias_{j}.fits",
        overwrite=True,
    )

    ## Dark frame
    hdu_dark = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=exp_time,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "Dark Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_dark[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_dark.writeto(
        f"data/{image_run}/Dark_{j}.fits",
        overwrite=True,
    )

    ## Flat frame
    site.sun_altitude = -4
    hdu_flat = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=0.5,  # seconds
        dateobs=dateobs,
        light=1,
        sources=cabaret.Sources.from_arrays(ra=[], dec=[], fluxes=[]),
        user_header={"FILTER": filter_band.name, "IMAGETYP": "Flat Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_flat[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_flat.writeto(
        f"data/{image_run}/Flat_{filter_band.name}_{j}.fits",
        overwrite=True,
    )
    site.sun_altitude = None

## Drift

In [124]:
drift_rate = 0.01  # arcseconds per second

image_run = f"{ra}_{dec}_s{seed}_n{n_imgs}_nc{n_imgs_cal}_{exp_time}s_r{readout_time}s_pds{if_pds}_drift{drift_rate}ass"
# Light frames
os.makedirs(f"data/{image_run}", exist_ok=True)
# Generate the FITS image for this filter
for j in range(n_imgs):
    dateobs = dateobs_0 + timedelta(seconds=j * (exp_time + readout_time))

    new_ra = ra + (drift_rate / 3600) * (j * (exp_time + readout_time))  # degrees
    new_dec = dec  # degrees
    ## Light frame
    hdu = observatory.generate_fits_image(
        ra=new_ra,  # degrees
        dec=new_dec,  # degrees
        exp_time=exp_time,  # seconds
        dateobs=dateobs,
        sources=sources,
        fwhm_multiplier=7,  # to determine the rendering radius around each star
        user_header={
            "FILTER": filter_band.name,
            "IMAGETYP": "Light Frame",
            "object": image_run,
        },
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu.writeto(
        f"data/{image_run}/{image_run}_{filter_band.name}_Light_{j}.fits",
        overwrite=True,
    )

# Calibration frames (bias, dark, flat)
os.makedirs(f"data/{image_run}", exist_ok=True)
for j in range(n_imgs_cal):
    dateobs = dateobs_0 + timedelta(seconds=j * (exp_time + readout_time))

    ## Bias frame
    hdu_bias = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=0,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "Bias Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_bias[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_bias.writeto(
        f"data/{image_run}/Bias_{j}.fits",
        overwrite=True,
    )

    ## Dark frame
    hdu_dark = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=exp_time,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "Dark Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_dark[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_dark.writeto(
        f"data/{image_run}/Dark_{j}.fits",
        overwrite=True,
    )

    ## Flat frame
    site.sun_altitude = -4
    hdu_flat = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=0.5,  # seconds
        dateobs=dateobs,
        light=1,
        sources=cabaret.Sources.from_arrays(ra=[], dec=[], fluxes=[]),
        user_header={"FILTER": filter_band.name, "IMAGETYP": "Flat Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_flat[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_flat.writeto(
        f"data/{image_run}/Flat_{filter_band.name}_{j}.fits",
        overwrite=True,
    )
    site.sun_altitude = None